### Create batch of prompts

#### Version -> tasks (13) x topics (49) x specialties (21) = 13'377
#### In a second step (in parse_gpt_response) add for 50% of the generated prompt an answer style specifications

In [3]:
import json
from tqdm import tqdm
import pandas as pd
import random

In [8]:
# Create batch of prompts

# Specify which model should be used to answer prompts
gpt_model = "gpt-4o"

# Path for input data and to store prompts
path_medical_ai_tasks = "../resources/medical_ai_tasks.json"
path_medical_topics = "../resources/medical_topics.json"
path_medical_professions = "../resources/medical_professions.json"
path_answer_styles = "../resources/answer_styles.json"
path_moove_examples = "../resources/generated_doctors_questions.jsonl"

output_path = "../results/batched_prompts.jsonl"

# Load data
with open(path_medical_ai_tasks, "r") as f:
    medical_ai_tasks = json.load(f)
with open(path_medical_topics, "r") as f:
    medical_topics = json.load(f)
with open(path_medical_professions, 'r') as f:
    professions_data = json.load(f)
with open(path_moove_examples, "r") as f:
    moove_examples = [json.loads(line) for line in f]
# Define number of few-shoot examples to be added
num_examples = 2

# Helper function to get uniformly at random num_entries moove examples
def get_random_entries(moove_examples, num_examples=2):
    """
    Returns a specified number of random entries from moove_examples without replacement.

    param moove_examples: List of JSON objects loaded from a .jsonl file.
    param num_entries: Number of random entries to return.
    return: List of randomly selected entries.
    """
    if len(moove_examples) < num_examples:
        raise ValueError(f"Not enough entries in moove_examples to select {num_examples} unique entries.")
    
    return random.sample(moove_examples, num_examples)



# prompt and context
content = "You are an assistant responsible for creating prompts that healthcare workers would ask a medical AI chatbot."
def get_prompt(task, description, additional_instruction, topic, profession, num_examples, example_1, example_2):
    # TODO: Make examples a list such that it has not to be hardcoded
    prompt = f'''Generate a prompt that a {profession} might ask an AI chatbot when tasked with "{task}" in the context of the medical topic "{topic}".
{task} is described as: {description}
To create a realistic prompt, follow these additional instructions: {additional_instruction}
Only include the generated prompt, adding extra details only if explicitly instructed. Focus solely on generating a realistic prompt a physician might ask a medical AI chatbot.
Below there are {num_examples} examples of real prompts that physicians have previously asked to the medical AI chatbot:
{example_1}
{example_2}
The examples provided are likely not directly related to "{task}" in the context of "{topic}", but they are representative of the format and style physicians use. The prompt you generate should align with the format and style of the {num_examples} examples provided above.'''
    return prompt


print("Creating a batch of prompts")
print(f"that can be processed by {gpt_model} in batch mode")

with open(output_path, 'w') as file:
    count = 0
    for ai_task in tqdm(medical_ai_tasks):
        task = ai_task["task"]
        description = ai_task["description"]
        max_token = ai_task["max_token"]
        additional_instruction = ai_task["additional_instruction"]
        for med_topic in medical_topics:
            topic = med_topic["topic"]
            for profession_entry in professions_data:
                profession = profession_entry["profession"]
                content = content
                example_1, example_2 = get_random_entries(moove_examples, num_examples)
                prompt = get_prompt(task=task, description=description, additional_instruction=additional_instruction,
                                    topic=topic, profession=profession, num_examples=num_examples,
                                    example_1 = example_1["question"], example_2= example_2["question"])
                line = {
                    "custom_id": str(count) + "-" + task + "-" + topic + "-" + profession,
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": {
                        "model": gpt_model,
                        "messages": [
                            {"role": "system", "content": content},
                            {"role": "user", "content": f"{prompt}"}
                        ],
                        "max_tokens": max_token
                    }
                }
                file.write(json.dumps(line) + '\n')
                count += 1

print(f"batch of {count} prompts saved to {output_path}")
print(f"See below an example prompt that will be processed by {gpt_model}:")
print("*******************************************************************")
print(prompt)
print("*******************************************************************")

Creating a batch of prompts
that can be processed by gpt-4o in batch mode


100%|██████████| 13/13 [00:00<00:00, 13.29it/s]

batch of 13377 prompts saved to ../results/batched_prompts.jsonl
See below an example prompt that will be processed by gpt-4o:
*******************************************************************
Generate a prompt that a Clinical Psychologist might ask an AI chatbot when tasked with "Health Policy Guidance" in the context of the medical topic "Preventive Medicine".
Health Policy Guidance is described as: Assisting with compliance to regulations and policies in healthcare.
To create a realistic prompt, follow these additional instructions: Provide a brief example of a healthcare regulation scenario, focusing on policy compliance.
Only include the generated prompt, adding extra details only if explicitly instructed. Focus solely on generating a realistic prompt a physician might ask a medical AI chatbot.
Below there are 2 examples of real prompts that physicians have previously asked to the medical AI chatbot:
I am a dermatologist in a US outpatient clinic, evaluating a 45-year-old Africa

In [7]:
# Visualy inspect prompts -> g

def sample_user_content(jsonl_path, n):
    """
    Reads a .jsonl file, randomly samples n entries, 
    and prints only the 'user' content for each sampled entry.
    
    :param jsonl_path: Path to the .jsonl file
    :param n: Number of entries to sample
    """
    # Load all lines (each line is valid JSON) into a list
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        data = [json.loads(line) for line in f]
    
    # Randomly sample n entries
    sampled_entries = random.sample(data, n)
    
    # Print only the user content for each sampled entry
    for entry in sampled_entries:
        messages = entry['body']['messages']
        
        # Extract and print the 'content' from messages where role='user'
        user_messages = [m['content'] for m in messages if m['role'] == 'user']
        for content in user_messages:
            print(content)
            print()  # Add a blank line for clarity


sample_user_content("../results/batched_prompts.jsonl", 10)

Generate a prompt that a Physician Assistant might ask an AI chatbot when tasked with "Patient Education" in the context of the medical topic "Toxicology".
Patient Education is described as: Providing information on diagnoses, treatments, and preventative care.
To create a realistic prompt, follow these additional instructions: Provide an example diagnosis that a healthcare worker wants to explain in simple terms to the patient.
Only include the generated prompt, adding extra details only if explicitly instructed. Focus solely on generating a realistic prompt a physician might ask a medical AI chatbot.
Below there are 2 examples of real prompts that physicians have previously asked to the medical AI chatbot:
A 32-year-old male patient, a professional musician, presents to your otolaryngology clinic with a 6-month history of progressive hearing loss in his left ear, accompanied by intermittent tinnitus and a sensation of ear fullness. He has no history of ear infections, trauma, or expo